In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split
print("All modules imported successfully")

All modules imported successfully


In [2]:
df=pd.read_csv('./Churn_Modelling.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.2 MB


In [4]:
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [5]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
df=df.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [7]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [8]:
le_gender=LabelEncoder()
df['Gender']=le_gender.fit_transform(df['Gender'])


In [9]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [10]:
ohe_geo=OneHotEncoder(handle_unknown='ignore')
geo_encoded=ohe_geo.fit_transform(df[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=ohe_geo.get_feature_names_out(['Geography']))
geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [11]:
df=pd.concat([df.drop(['Geography'],axis=1),geo_encoded_df],axis=1)

In [12]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [13]:
X=df.drop('EstimatedSalary',axis=1)
y=df['EstimatedSalary']

In [14]:
# Save encoder and scaler objects for future use
with open('le_gender.pkl', 'wb') as f:
    pickle.dump(le_gender, f)

with open('ohe_geo.pkl', 'wb') as f:
    pickle.dump(ohe_geo, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(StandardScaler(), f)

    

In [15]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [16]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

## ANN Regression Model

In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [19]:
## Build model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1,activation='linear')
])

# Compile model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [20]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

log_dir='regressionlogs/fit/' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [21]:
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [22]:
history=model.fit(
X_train,y_train,validation_data=(X_test,y_test),epochs=100,batch_size=32,callbacks=[early_stopping_callback,tensorboard_callback]
)

Epoch 1/100


250/250 [==============================] - 9s 15ms/step - loss: 100377.5078 - mae: 100377.5078 - val_loss: 98515.3438 - val_mae: 98515.3438
Epoch 2/100
250/250 [==============================] - 3s 12ms/step - loss: 99634.5156 - mae: 99634.5156 - val_loss: 97025.3750 - val_mae: 97025.3750
Epoch 3/100
250/250 [==============================] - 3s 11ms/step - loss: 97061.8984 - mae: 97061.8984 - val_loss: 93248.7734 - val_mae: 93248.7734
Epoch 4/100
250/250 [==============================] - 3s 12ms/step - loss: 92011.1172 - mae: 92011.1172 - val_loss: 86945.6250 - val_mae: 86945.6250
Epoch 5/100
250/250 [==============================] - 3s 12ms/step - loss: 84606.9141 - mae: 84606.9141 - val_loss: 78716.3125 - val_mae: 78716.3125
Epoch 6/100
250/250 [==============================] - 2s 9ms/step - loss: 75785.4688 - mae: 75785.4688 - val_loss: 69997.3203 - val_mae: 69997.3203
Epoch 7/100
250/250 [==============================] - 2s 7ms/step - loss: 67063.0000 - mae: 6706

In [23]:
%load_ext tensorboard

In [25]:
%tensorboard --logdir=regressionlogs/fit

In [26]:
## Evaluate the model
loss, mae = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}, Test MAE: {mae}")

63/63 [==============================] - 0s 6ms/step - loss: 50267.5391 - mae: 50267.5391
Test Loss: 50267.5390625, Test MAE: 50267.5390625
